# Homework 5 - API Practice

## PART 2: Genius API

### a. Load credentials and Last.fm tracks

This block loads the Genius access token from `.env` and creates the authorization header for API requests. It initializes LyricsGenius to retrieve lyrics without section headers, with a timeout and retry setting.

In [1]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv
from lyricsgenius import Genius

# load environment variables
load_dotenv()

# set up Genius API client and headers for requests
token = os.environ["GENIUS_ACCESS_TOKEN"]
headers = {"Authorization": f"Bearer {token}"}
genius = Genius(token, remove_section_headers=True, timeout=20, retries=1)

# load lastfm data
lastfm = pd.read_csv("../data/lastfm.csv")

### b. Retrieve lyrics and songwriting credits

This block searches Genius for each Last.fm song using its title and “Jay Chou.” It filters the search results to retain songs whose primary artist is Jay Chou, then uses the first remaining result’s ID to retrieve lyrics and songwriting credits.

Each record stores the Last.fm album and song names together with the retrieved lyrics, writers, and producers. Retaining the Last.fm names provides consistent keys for the later merge. Also, multiple writer or producer names are joined with semicolons so they can fit into a single field.

In [2]:
rows = []

# iterate through lastfm tracks and fetch Genius data
for track in lastfm.itertuples(index=False):
    # search for the track on Genius
    params = {"q": f"{track.song_name} Jay Chou"}
    response = requests.get("https://api.genius.com/search", headers=headers, params=params)
    hits = response.json()["response"]["hits"]

    # filter out romanization and translation pages
    jay_hits = [hit["result"] for hit in hits if hit["result"]["primary_artist"]["name"] == "周杰倫 (Jay Chou)"]
    song = genius.search_song(song_id=jay_hits[0]["id"])

    # fetch the song information from the Genius API for the filtered hit
    response = requests.get(f"https://api.genius.com/songs/{jay_hits[0]['id']}", headers=headers)
    info = response.json()["response"]["song"]

    # get the relevant song information and append it to the rows list
    rows.append({
        "album": track.album,
        "song_name": track.song_name,
        "lyrics": song.lyrics,
        "writer_artists": "; ".join(artist["name"] for artist in info.get("writer_artists", [])),
        "producer_artists": "; ".join(artist["name"] for artist in info.get("producer_artists", []))
    })

This block creates a DataFrame containing the album, song name, lyrics, writer credits, and producer credits. Displaying the first ten rows provides an initial check of the collected information and column structure.

In [3]:
# create a DataFrame from the collected Genius data
genius_data = pd.DataFrame(rows, columns=["album", "song_name", "lyrics", "writer_artists", "producer_artists"])
genius_data.head(10)

,album,song_name,lyrics,writer_artists,producer_artists
0,11月的蕭邦,夜曲,一群嗜血的螞蟻 被腐肉所吸引\n我面無表情 看孤獨的風景\n失去妳 愛恨開始分明\n失去妳 ...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
1,11月的蕭邦,藍色風暴,亙古長夜 善惡交接 我終於懂得流眼淚\n洪荒世界 百分之七十都是水 湛藍的曠野\n在海平面渴...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
2,11月的蕭邦,髮如雪,狼牙月 伊人憔悴\n我舉杯 飲盡了風雪\n是誰打翻前世櫃 惹塵埃是非\n緣字訣 幾番輪迴\n...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
3,11月的蕭邦,黑色毛衣,一件黑色毛衣 兩個人的回憶\n雨過之後 更難忘記 忘記我還愛你\n你不用在意 流淚也只是剛好...,周杰倫 (Jay Chou),周杰倫 (Jay Chou)
4,11月的蕭邦,四面楚歌,我的生活像 拍了一齣戲\n有超多導演跟編劇 只說了台詞一句\n而他們配了八百個語氣 操控著我...,周杰倫 (Jay Chou),周杰倫 (Jay Chou)
5,11月的蕭邦,楓,烏雲在我們心裏擱下一塊陰影\n我聆聽沉寂已久的心情\n清晰透明 就像美麗的風景\n總在回憶裏...,Devon Song,周杰倫 (Jay Chou)
6,11月的蕭邦,浪漫手機,輕輕放 我就是卸不下對你的喜歡\n原來愛會慢慢增加重量\n想關上這城市所有的燈光\n黑暗中專...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
7,11月的蕭邦,逆鱗,而生命 (對每個人)\n都不公平 (也沒道理)\n只能撲向 (泥濘迎向)\n那陣驟雨 (由不...,黃俊郎 (Alang Huang),周杰倫 (Jay Chou)
8,11月的蕭邦,麥芽糖,這蜿蜒的微笑擁抱山丘\n溪流跟風唱起歌\n我像田園詩人般解讀眼前的生活\n麥田彎腰低頭在垂釣...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
9,11月的蕭邦,珊瑚海,海平面遠方開始陰霾\n悲傷要怎麼平靜純白\n我的臉上 始終挾帶\n一抹淺淺的無奈\n你用唇語...,方文山 (Vincent Fang),周杰倫 (Jay Chou)


This block saves the DataFrame to a csv file for the combining and EDA step.

In [4]:
# save the collected Genius data to a csv file
genius_data.to_csv("../data/genius.csv", index=False)